# 📖 Notebook 1: Fan-Out on Write vs Fan-Out on Read

The single biggest design decision in a News Feed system is **when** you assemble each user's feed.  
There are two opposite approaches — and most production systems use a mix of both.

## Learning Objectives

By the end of this notebook, you'll understand:
- What fan-out on **read** means and when it's appropriate
- What fan-out on **write** means and why it's faster for readers
- The trade-offs: write amplification, storage, latency
- How a **hybrid** approach handles celebrities vs normal users

## 🛠️ Setup

Start the infrastructure first:

```bash
cd system-designs/fb-news-feed
docker-compose up -d
```

### Visualization Tools

- **Adminer** (PostgreSQL GUI): http://localhost:8080  
  Login: System `PostgreSQL`, Server `postgres`, User `demo`, Password `demo`, Database `newsfeed_demo`
- **RedisInsight** (Redis GUI): http://localhost:5540  
  Click "Add Redis Database" → Host `redis`, Port `6379`

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
import psycopg2
import psycopg2.extras
import time

# Database connection settings
DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "newsfeed_demo",
    "user": "demo",
    "password": "demo"
}

def get_db():
    return psycopg2.connect(**DB_CONFIG)

# Quick connection test
try:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM users")
    print(f"✅ Connected — {cur.fetchone()[0]} users in database")
    cur.execute("SELECT COUNT(*) FROM follows")
    print(f"   {cur.fetchone()[0]} follow relationships")
    cur.execute("SELECT COUNT(*) FROM posts")
    print(f"   {cur.fetchone()[0]} posts")
    conn.close()
except Exception as e:
    print(f"❌ Connection failed: {e}")
    print("   Run: docker-compose up -d")

## 🤔 The Core Problem

When User A opens their News Feed, we need to show them recent posts from everyone they follow.  
The question is: **when do we collect those posts?**

```
Imagine Alice follows 500 people. Each person posts ~3 times/day.
That's 1,500 posts we might need to look through — just for Alice.

Now imagine 2 BILLION users doing this simultaneously.
```

There are two fundamental strategies:

## Strategy 1: Fan-Out on Read ("Pull Model")

**When the user opens their feed**, we:
1. Look up everyone they follow
2. Fetch recent posts from each of those users
3. Merge and sort by time
4. Return the top N

```
User opens feed
      │
      ▼
┌─────────────┐     ┌──────────────┐
│ Get follows  │────►│ 500 followees │
└─────────────┘     └──────┬───────┘
                           │
                    For EACH followee:
                    fetch recent posts
                           │
                           ▼
                    Merge + Sort + Return
```

Let's implement this and measure it.

In [ ]:
def fan_out_on_read(user_id: int, limit: int = 20) -> list:
    """
    Build a user's feed at READ time.
    
    Step 1: Find everyone this user follows.
    Step 2: For each followee, get their recent posts.
    Step 3: Merge all posts, sort by time, return top N.
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    # Step 1: Who does this user follow?
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s",
        (user_id,)
    )
    followee_ids = [row["followee_id"] for row in cur.fetchall()]
    
    if not followee_ids:
        conn.close()
        return []
    
    # Step 2 + 3: Get posts from all followees, sorted by time
    cur.execute(
        """
        SELECT p.id, p.content, p.created_at,
               u.username, u.display_name
        FROM posts p
        JOIN users u ON u.id = p.author_id
        WHERE p.author_id = ANY(%s)
        ORDER BY p.created_at DESC
        LIMIT %s
        """,
        (followee_ids, limit)
    )
    feed = cur.fetchall()
    conn.close()
    return feed

# Try it for user 1
start = time.time()
feed = fan_out_on_read(user_id=1, limit=10)
elapsed = (time.time() - start) * 1000

print(f"⏱️  Fan-out on READ took {elapsed:.1f} ms")
print(f"📰 Feed for user 1 ({len(feed)} posts):")
print("-" * 70)
for post in feed:
    print(f"  [{post['created_at']:%H:%M}] @{post['username']}: {post['content'][:50]}")

### ⚠️ Why Fan-Out on Read Gets Slow

Fan-out on read works fine for small numbers. But imagine:
- User follows **5,000** people (Facebook's friend limit)
- We need results in **< 500 ms**
- We're doing this for **millions of concurrent users**

Every feed request hits the database with a potentially huge `IN (...)` query.  
The database becomes the bottleneck.

In [ ]:
# Let's measure how fan-out on read scales with number of followees.
# We'll simulate users following different numbers of people.

conn = get_db()
cur = conn.cursor()

# Check how many people each user follows
cur.execute("""
    SELECT f.follower_id, COUNT(*) as follow_count
    FROM follows f
    GROUP BY f.follower_id
    ORDER BY follow_count DESC
    LIMIT 10
""")

print("👥 Top 10 users by follow count:")
print(f"{'User ID':>8}  {'Follows':>8}")
print("-" * 20)
for row in cur.fetchall():
    print(f"{row[0]:>8}  {row[1]:>8}")

conn.close()

# Now benchmark fan-out on read for several users
print("\n⏱️  Fan-out on read latency per user:")
print(f"{'User':>6}  {'Follows':>8}  {'Latency':>10}")
print("-" * 30)

for uid in [1, 5, 10, 25, 50]:
    conn = get_db()
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM follows WHERE follower_id = %s", (uid,))
    n_follows = cur.fetchone()[0]
    conn.close()
    
    times = []
    for _ in range(20):
        t0 = time.time()
        fan_out_on_read(uid, limit=20)
        times.append((time.time() - t0) * 1000)
    avg = sum(times) / len(times)
    print(f"{uid:>6}  {n_follows:>8}  {avg:>8.1f} ms")

print("\n💡 Latency grows with the number of followees — at scale this is a problem.")

## Strategy 2: Fan-Out on Write ("Push Model")

**When a post is created**, we immediately push it into every follower's precomputed feed.

```
User creates a post
      │
      ▼
┌─────────────────┐     ┌────────────────────┐
│ Get followers    │────►│ 1,000 followers     │
└─────────────────┘     └──────┬─────────────┘
                               │
                    For EACH follower:
                    INSERT post into their feed
                               │
                               ▼
                    Done! Feed is pre-built.
```

Reading the feed is now just a simple query on the precomputed table.

In [ ]:
def fan_out_on_write(author_id: int, content: str) -> dict:
    """
    Create a post and push it to all followers' precomputed feeds.
    This is the WRITE side of fan-out on write.
    """
    conn = get_db()
    cur = conn.cursor()
    
    # Step 1: Create the post
    cur.execute(
        "INSERT INTO posts (author_id, content) VALUES (%s, %s) RETURNING id, created_at",
        (author_id, content)
    )
    post_id, created_at = cur.fetchone()
    
    # Step 2: Find all followers of this author
    cur.execute(
        "SELECT follower_id FROM follows WHERE followee_id = %s",
        (author_id,)
    )
    follower_ids = [row[0] for row in cur.fetchall()]
    
    # Step 3: Push the post into each follower's precomputed feed
    if follower_ids:
        psycopg2.extras.execute_values(
            cur,
            """
            INSERT INTO precomputed_feed (user_id, post_id, post_author_id, post_created_at)
            VALUES %s
            ON CONFLICT DO NOTHING
            """,
            [(fid, post_id, author_id, created_at) for fid in follower_ids]
        )
    
    conn.commit()
    conn.close()
    
    return {
        "post_id": post_id,
        "followers_updated": len(follower_ids)
    }


def read_precomputed_feed(user_id: int, limit: int = 20) -> list:
    """
    Read a user's feed from the precomputed table.
    This is the READ side — it's just one simple query!
    """
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    cur.execute(
        """
        SELECT pf.post_id, p.content, pf.post_created_at,
               u.username, u.display_name
        FROM precomputed_feed pf
        JOIN posts p ON p.id = pf.post_id
        JOIN users u ON u.id = pf.post_author_id
        WHERE pf.user_id = %s
        ORDER BY pf.post_created_at DESC
        LIMIT %s
        """,
        (user_id, limit)
    )
    feed = cur.fetchall()
    conn.close()
    return feed

# Demo: Create a new post and push to followers
result = fan_out_on_write(
    author_id=1,
    content="Just posted from notebook! Fan-out on write demo."
)
print(f"✍️  Created post #{result['post_id']}")
print(f"📤 Pushed to {result['followers_updated']} followers' feeds")

In [ ]:
# Compare READ latency: fan-out on read vs precomputed feed

print("⏱️  Reading feed — Fan-Out on Read vs Precomputed:")
print("=" * 55)

for uid in [1, 10, 25]:
    # Fan-out on read
    times_read = []
    for _ in range(50):
        t0 = time.time()
        fan_out_on_read(uid)
        times_read.append((time.time() - t0) * 1000)
    
    # Precomputed feed
    times_precomp = []
    for _ in range(50):
        t0 = time.time()
        read_precomputed_feed(uid)
        times_precomp.append((time.time() - t0) * 1000)
    
    avg_read = sum(times_read) / len(times_read)
    avg_precomp = sum(times_precomp) / len(times_precomp)
    
    print(f"\n  User {uid}:")
    print(f"    Fan-out on read:  {avg_read:.2f} ms")
    print(f"    Precomputed:      {avg_precomp:.2f} ms")
    if avg_precomp > 0:
        print(f"    Speedup:          {avg_read / avg_precomp:.1f}×")

print("\n💡 Precomputed feeds trade write-time work for blazing fast reads.")

## ⚠️ The Celebrity Problem (Write Amplification)

Fan-out on write works great for regular users, but what about celebrities?

```
Justin Bieber has 90 million followers.
Every time he posts, we'd need to write to 90 MILLION feed entries.
That's enormous write amplification!
```

Let's see the write cost for different follower counts.

In [ ]:
# Check follower counts for our celebrities vs regular users

conn = get_db()
cur = conn.cursor()

cur.execute("""
    SELECT u.id, u.display_name,
           COUNT(f.follower_id) as follower_count
    FROM users u
    LEFT JOIN follows f ON f.followee_id = u.id
    GROUP BY u.id, u.display_name
    ORDER BY follower_count DESC
    LIMIT 10
""")

print("👥 Users by follower count:")
print(f"{'ID':>4}  {'Name':<25}  {'Followers':>10}")
print("-" * 45)
for row in cur.fetchall():
    print(f"{row[0]:>4}  {row[1]:<25}  {row[2]:>10}")
conn.close()

# Demonstrate write cost
print("\n📝 Write cost when posting (fan-out on write):")
print("-" * 50)

# Time the write for a regular user vs celebrity
for author_id, label in [(5, "Regular user"), (51, "Celebrity Alice")]:
    times = []
    for i in range(5):
        t0 = time.time()
        fan_out_on_write(author_id, f"Benchmark post {i} from {label}")
        times.append((time.time() - t0) * 1000)
    avg = sum(times) / len(times)
    print(f"  {label}: {avg:.1f} ms per post")

print("\n💡 More followers = slower writes. Celebrities are expensive!")

## 🏆 Strategy 3: The Hybrid Approach

Production systems like Facebook and Twitter use a **hybrid** approach:

| User Type | Strategy | Why |
|-----------|----------|-----|
| Normal (< 10K followers) | Fan-out on **write** | Fast reads, manageable write cost |
| Celebrity (> 10K followers) | Fan-out on **read** | Avoid millions of writes per post |

When a user reads their feed:
1. Grab their **precomputed feed** (fan-out on write results)
2. Fetch recent posts from **celebrity followees** on the fly
3. **Merge** the two lists, sort by time, return top N

This gives us the best of both worlds!

In [ ]:
CELEBRITY_THRESHOLD = 30  # In production this would be ~10,000+

def get_celebrity_ids() -> set:
    """Find users with more followers than the threshold."""
    conn = get_db()
    cur = conn.cursor()
    cur.execute("""
        SELECT followee_id
        FROM follows
        GROUP BY followee_id
        HAVING COUNT(*) > %s
    """, (CELEBRITY_THRESHOLD,))
    ids = {row[0] for row in cur.fetchall()}
    conn.close()
    return ids


def hybrid_read_feed(user_id: int, limit: int = 20) -> list:
    """
    Hybrid approach: merge precomputed feed with on-the-fly celebrity posts.
    """
    celebrity_ids = get_celebrity_ids()
    conn = get_db()
    cur = conn.cursor(cursor_factory=psycopg2.extras.RealDictCursor)
    
    # Part A: Get precomputed feed (excludes celebrity posts)
    cur.execute(
        """
        SELECT pf.post_id AS id, p.content, pf.post_created_at AS created_at,
               u.username, u.display_name, 'precomputed' AS source
        FROM precomputed_feed pf
        JOIN posts p ON p.id = pf.post_id
        JOIN users u ON u.id = pf.post_author_id
        WHERE pf.user_id = %s
          AND pf.post_author_id != ALL(%s)
        ORDER BY pf.post_created_at DESC
        LIMIT %s
        """,
        (user_id, list(celebrity_ids), limit)
    )
    precomputed = cur.fetchall()
    
    # Part B: Fan-out on read for celebrities this user follows
    cur.execute(
        "SELECT followee_id FROM follows WHERE follower_id = %s AND followee_id = ANY(%s)",
        (user_id, list(celebrity_ids))
    )
    my_celebrity_followees = [row["followee_id"] for row in cur.fetchall()]
    
    celebrity_posts = []
    if my_celebrity_followees:
        cur.execute(
            """
            SELECT p.id, p.content, p.created_at,
                   u.username, u.display_name, 'fan-out-read' AS source
            FROM posts p
            JOIN users u ON u.id = p.author_id
            WHERE p.author_id = ANY(%s)
            ORDER BY p.created_at DESC
            LIMIT %s
            """,
            (my_celebrity_followees, limit)
        )
        celebrity_posts = cur.fetchall()
    
    conn.close()
    
    # Part C: Merge both lists, sort by time, return top N
    merged = list(precomputed) + list(celebrity_posts)
    merged.sort(key=lambda x: x["created_at"], reverse=True)
    return merged[:limit]


# Try the hybrid approach
celeb_ids = get_celebrity_ids()
print(f"🌟 Celebrity IDs (>{CELEBRITY_THRESHOLD} followers): {celeb_ids}")
print()

feed = hybrid_read_feed(user_id=1, limit=10)
print(f"📰 Hybrid feed for user 1 ({len(feed)} posts):")
print("-" * 80)
for post in feed:
    source = post['source']
    icon = '📌' if source == 'precomputed' else '🌟'
    print(f"  {icon} [{post['created_at']}] @{post['username']}: {post['content'][:45]}")
    print(f"       source: {source}")

## 📊 Side-by-Side Comparison

Let's put all three strategies head-to-head.

In [ ]:
uid = 1

strategies = [
    ("Fan-out on Read",  lambda: fan_out_on_read(uid)),
    ("Precomputed Feed", lambda: read_precomputed_feed(uid)),
    ("Hybrid",           lambda: hybrid_read_feed(uid)),
]

print("📊 Read Latency Comparison (50 iterations each):")
print("=" * 50)

for name, fn in strategies:
    times = []
    for _ in range(50):
        t0 = time.time()
        fn()
        times.append((time.time() - t0) * 1000)
    avg = sum(times) / len(times)
    p95 = sorted(times)[int(len(times) * 0.95)]
    print(f"  {name:<22} avg={avg:>6.2f} ms   p95={p95:>6.2f} ms")

print()
print("📋 Summary:")
print("┌───────────────────────┬───────────────┬───────────────┬─────────────────────┐")
print("│ Strategy              │ Read Speed    │ Write Speed   │ Best For            │")
print("├───────────────────────┼───────────────┼───────────────┼─────────────────────┤")
print("│ Fan-out on Read       │ ❌ Slow       │ ✅ Fast       │ Few follows         │")
print("│ Fan-out on Write      │ ✅ Fast       │ ❌ Slow       │ Few followers       │")
print("│ Hybrid                │ ✅ Fast       │ ✅ Balanced   │ Production systems  │")
print("└───────────────────────┴───────────────┴───────────────┴─────────────────────┘")

## 🧹 Cleanup

In [ ]:
# Remove the benchmark posts we created
conn = get_db()
cur = conn.cursor()
cur.execute("DELETE FROM precomputed_feed WHERE post_id IN (SELECT id FROM posts WHERE content LIKE 'Benchmark post%')")
cur.execute("DELETE FROM precomputed_feed WHERE post_id IN (SELECT id FROM posts WHERE content LIKE '%Fan-out on write demo%')")
cur.execute("DELETE FROM posts WHERE content LIKE 'Benchmark post%'")
cur.execute("DELETE FROM posts WHERE content LIKE '%Fan-out on write demo%'")
conn.commit()
print(f"🧹 Cleaned up benchmark data")
conn.close()

## 📚 Summary

### Key Takeaways

1. **Fan-out on Read** builds the feed when you open it — simple but slow at scale
2. **Fan-out on Write** pre-builds the feed when a post is created — fast reads but expensive writes
3. **Hybrid** uses fan-out on write for normal users and fan-out on read for celebrities
4. Production systems (Facebook, Twitter/X, Instagram) all use the hybrid approach

### Interview Tips

- Start with the naive approach (fan-out on read), then explain why it's slow
- Introduce fan-out on write as the optimisation
- Bring up the celebrity problem unprompted — interviewers love this
- Propose the hybrid solution as the production-grade answer

### Next Up

In **Notebook 2**, we'll explore **News Feed Ranking** — how to move beyond simple chronological order to surface the most relevant posts first.